# ETTh1 rerun with per-window error logging

Reruns the committed ETTh1 protocol (CI + CD x 4 horizons x 5 seeds) with one
addition: per-test-window squared/absolute error is logged and joined by
`window_start` to the already-committed `etth1_coupling_partition.csv`
(`src/analysis/partition_etth1_coupling.py`), enabling the subgroup analysis —
does CD's accuracy relative to CI differ between high-coupling and low-coupling
test windows?

**Protocol: the committed ETTh1 protocol verbatim** (from `train_etth1.ipynb`):
- Real ETTh1, 7 variates, seq_len=512, horizons {96, 192, 336, 720}.
- Standard split by timestep: train [0, 8640), val [8640, 11520), test
  [11520, 17420); z-score fit on train only.
- Batch 32 for BOTH modes (removes the original step-count confound); warmup
  10 epochs for BOTH modes (removes the original LR-schedule confound);
  min_epochs=15 before early stopping can fire; patience 10; max_epochs 100.
- AdamW lr=1e-4, wd=1e-4, grad clip 1.0, AMP; cosine decay to a hard 0 floor
  (NOT the 1e-6 floor used by the boundary protocol — kept verbatim per family).
- Eval WITHOUT autocast (differs from the boundary protocol's eval-under-
  autocast convention; each family keeps its own committed eval mode).

**Models:** from the attached private dataset `b1-block-attn` (`models.py`,
verbatim from branch `revision/reviewer-yc7L`) — the same committed,
hash-verified module, not the standalone classes inlined in
`train_etth1.ipynb`. `build_model()`'s defaults (patch_size=16, stride=8,
d_model=64, n_heads=8, n_layers=3, dropout=0.2) already equal the ETTh1
config, so this is architecture-identical to the original run — a change of
*source*, not of protocol. **PROPOSED — confirm before first launch.**

**Engine: per-epoch atomic checkpoint+resume, config-driven run list,
and idempotent registry, unchanged in mechanism.** Delta from
`train_boundary_p4.ipynb`:
- Data/model/schedule/eval swapped for the ETTh1 protocol above (real data,
  not synthetic; no autocast at eval; MIN_EPOCHS gate added to early
  stopping; no-floor cosine; `drop_last=False` on the train loader, matching
  `train_etth1.ipynb`, not the boundary protocol's `drop_last=True`).
- Checkpoint/registry key is (pred_len, mode, seed), not (gamma, mode, seed).
- **New:** a second, append-only CSV logs per-test-window squared/absolute
  error, keyed by (mode, pred_len, seed, window_start), written once per
  completed run alongside the aggregate row.
- **New, declared deviation, safety-only:** a non-finite validation-loss
  guard stops a run cleanly instead of letting NaNs propagate into a
  checkpoint (present in the boundary engine, absent from the original
  `train_etth1.ipynb`). Never fires on a finite trajectory.

**`window_start` convention — UNVERIFIED, confirm against
`partition_etth1_coupling.py` before running the subgroup join:** logged
here as the absolute index into the full ETTh1 array (`VAL_END + i` for the
i-th 0-indexed test window). Test-window counts below reproduce the
committed partition's reported per-horizon sizes exactly (2647+2646=5293 at
H=96, 2335+2334=4669 at H=720), which supports this being the right split
and ordering — but not necessarily the same zero-point convention the
partition script itself used. That script's source has not been
cross-checked directly.

**Pricing: not yet measured for this notebook.** `EPOCH_EST_S` below is a
placeholder that self-corrects from the first session's own measured epoch
times, same mechanism as the session budget check — treat the first session
as the de facto probe.

**Session budget / multi-session resume:** identical procedure — attach
the previous version's output as an input on a new version, Run All.

**Launch:** new notebook, GPU T4, attach `b1-block-attn`, `config_etth1_b4.json`,
and an ETTh1 dataset containing `ETTh1.csv` (first launch: `alaaelmor/ettsmall`;
`load_etth1()` searches for it by filename under `/kaggle/input`, so any
dataset containing `ETTh1.csv` works; no path to match manually). Plus the
previous session's output from session 2 onward.

In [ ]:
# ── Cell 1: Environment ──────────────────────────────────────────────────────
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import gc
import json
import math
import random
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, TensorDataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU:  {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def free_cuda() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
# ── Cell 2: Committed-module import with tripwire asserts ───────────────────
# Model code is NOT inlined. It comes from the attached private Kaggle dataset
# b1-block-attn (the same shared dataset the other notebooks use), whose files
# are verbatim copies from branch revision/reviewer-yc7L. models_cd_block.py is
# present in this dataset but unused here (CI/CD only, no CD_Block for ETTh1);
# the assertion is kept identical so the same trusted, hash-verified source is
# confirmed rather than special-cased.
import hashlib
import sys

EXPECTED_FILES = {"models.py", "models_cd_block.py"}

INPUT_ROOT = Path("/kaggle/input")
# Diagnostic: list where attached content actually lives before any assertion
# can fail, so a failure message is immediately actionable without an
# attach-rerun cycle to discover it. Listing INPUT_ROOT's immediate children
# is not useful here -- Kaggle's real mount nests dataset-type inputs under
# /kaggle/input/datasets/<owner>/<slug>/ (confirmed 06 Aug against real
# attached-input paths), so the top level is just one "datasets" folder.
# Every leaf directory that actually contains a file is what matters, at
# whatever depth Kaggle puts it -- and it's also what every rglob-based
# lookup below depends on, so this print is a direct preview of what those
# lookups will see.
_leaf_dirs = sorted({p.parent for p in INPUT_ROOT.rglob("*") if p.is_file()})
print(f"Attached input directories (containing files) under {INPUT_ROOT}:")
for _d in _leaf_dirs:
    print(f"  {_d}")

_hits = sorted(INPUT_ROOT.rglob("models.py"))
assert len(_hits) == 1, (
    f"Expected exactly one attached dataset containing models.py; found {_hits}. "
    "Attach b1-block-attn and nothing else that carries a models.py."
)
MODULE_DIR = _hits[0].parent
_py_files = {p.name for p in MODULE_DIR.glob("*.py")}
assert _py_files == EXPECTED_FILES, (
    f"Dataset must contain exactly {sorted(EXPECTED_FILES)}; found {sorted(_py_files)}."
)

sys.path.insert(0, str(MODULE_DIR))
import models as M  # noqa: E402

assert callable(getattr(M, "build_model", None)), "models.py must expose build_model()"
for _f in sorted(EXPECTED_FILES):
    _h = hashlib.sha256((MODULE_DIR / _f).read_bytes()).hexdigest()[:16]
    print(f"  {_f}: sha256[:16]={_h}")
print(f"Committed modules loaded from {MODULE_DIR}")


In [ ]:
# ── Cell 3: Assignment config from the shared dataset ───────────────────────
# The run list is data, not code, matching the config-driven convention.
_cfg_hits = sorted(INPUT_ROOT.rglob("config_etth1_b4.json"))
assert len(_cfg_hits) == 1, (
    f"Expected exactly one config_etth1_b4.json across attached datasets; "
    f"found {_cfg_hits}. Attach the config dataset (and only one version of it)."
)
CFG_PATH = _cfg_hits[0]
CFG = json.loads(CFG_PATH.read_text())

for _k in ("assignment", "batch_size", "runs"):
    assert _k in CFG, f"config_etth1_b4.json missing key: {_k}"
for _r in CFG["runs"]:
    for _k in ("pred_len", "mode", "seeds"):
        assert _k in _r, f"run entry missing key {_k}: {_r}"
    assert _r["mode"] in ("CI", "CD"), f"Unknown mode in config: {_r['mode']}"
    assert _r["pred_len"] in (96, 192, 336, 720), f"Unknown pred_len in config: {_r['pred_len']}"

SESSION_BUDGET_S = float(CFG.get("session_budget_hours", 11.0)) * 3600.0

RUN_LIST: list[tuple[int, str, int]] = [
    (int(r["pred_len"]), str(r["mode"]), int(s))
    for r in CFG["runs"] for s in r["seeds"]
]
print(f"Assignment: {CFG['assignment']}  (config sha256[:16]="
      f"{hashlib.sha256(CFG_PATH.read_bytes()).hexdigest()[:16]})")
print(f"Run list ({len(RUN_LIST)} runs, in execution order):")
for h, m, s in RUN_LIST:
    print(f"  pred_len={h} mode={m} seed={s}")


In [ ]:
# ── Cell 4: ETTh1 protocol — data (verbatim from the committed train_etth1.ipynb) ──
# Real ETTh1, NOT synthetic. Standard non-overlapping split by timestep,
# z-score fit on train only. Committed elsewhere as results_etth1.csv; this
# rerun's aggregate numbers are NOT pooled with that CSV — fresh sweep
# of the same committed protocol, reported and analysed separately.

SEQ_LEN: int = 512
PRED_LENS: list[int] = [96, 192, 336, 720]

PATCH_SIZE:   int = 16
PATCH_STRIDE: int = 8
D_MODEL:  int   = 64
N_HEADS:  int   = 8
N_LAYERS: int   = 3
DROPOUT:  float = 0.2

BATCH_SIZE = int(CFG["batch_size"])
assert BATCH_SIZE == 32, f"Committed ETTh1 protocol uses batch=32 for both modes; config says {BATCH_SIZE}."
LR, WEIGHT_DECAY, GRAD_CLIP = 1e-4, 1e-4, 1.0
WARMUP_EPOCHS, MAX_EPOCHS, PATIENCE, MIN_EPOCHS = 10, 100, 10, 15

TRAIN_END: int = 8640
VAL_END:   int = 11520   # 8640 + 2880

SCHEMA_MAIN = ["mode", "pred_len", "seed", "test_mse", "test_mae", "best_epoch",
               "batch_size", "steps_per_epoch", "total_steps_to_best",
               "warmup_epochs", "min_epochs", "max_epochs"]
SCHEMA_WINDOWS = ["mode", "pred_len", "seed", "window_start", "sq_err", "abs_err"]

# Committed-CSV truths, derived from ETTh1's standard 17,420-row length and the
# split above; cross-checked against partition_etth1_coupling.py's reported
# per-horizon test-window counts (2647+2646=5293 at H=96, 2335+2334=4669 at
# H=720 — both reproduced exactly below).
EXPECTED_T_TOTAL       = 17_420
EXPECTED_TRAIN_WINDOWS = {96: 8033, 192: 7937, 336: 7793, 720: 7409}
EXPECTED_TEST_WINDOWS  = {96: 5293, 192: 5197, 336: 5053, 720: 4669}
EXPECTED_SPE_B32       = {96: 252,  192: 249,  336: 244,  720: 232}


def load_etth1():
    # Discovered the same way the notebook already trusts its other two
    # required inputs (models.py, config_etth1_b4.json): rglob under
    # /kaggle/input, not a guessed path list. A hardcoded candidate list
    # broke on the first real dataset -- alaaelmor/ettsmall mounts at
    # /kaggle/input/datasets/alaaelmor/ettsmall/ETTh1.csv in this project's
    # sessions (Kaggle nests dataset-type inputs under /kaggle/input/datasets/
    # <owner>/<slug>/, confirmed 06 Aug against real attached-input paths),
    # which matched none of a prior guessed list. rglob is depth-agnostic,
    # so it does not matter whether Kaggle changes that nesting again.
    _hits = sorted(INPUT_ROOT.rglob("ETTh1.csv"))
    if len(_hits) == 1:
        p = _hits[0]
    elif len(_hits) > 1:
        raise FileNotFoundError(
            f"Multiple ETTh1.csv found across attached datasets: {_hits}. "
            "Attach exactly one ETTh1 dataset.")
    elif Path("/kaggle/working/ETTh1.csv").exists():
        p = Path("/kaggle/working/ETTh1.csv")
    else:
        raise FileNotFoundError(
            "ETTh1.csv not found under /kaggle/input (searched recursively) or at "
            "/kaggle/working/ETTh1.csv. Attach a Kaggle dataset containing it, or "
            "copy it to /kaggle/working/ETTh1.csv."
        )
    df = pd.read_csv(p)
    num_cols = df.select_dtypes(include=np.number).columns.tolist()
    print(f"Loaded ETTh1 from {p}: {len(df)} rows, {len(num_cols)} variates")
    return df[num_cols].values.astype(np.float64)


def split_normalise_etth1(data):
    train = data[:TRAIN_END]
    val   = data[TRAIN_END:VAL_END]
    test  = data[VAL_END:]
    mean  = train.mean(axis=0, keepdims=True)
    std   = train.std(axis=0, keepdims=True)
    std   = np.where(std == 0, 1.0, std)
    return (train - mean) / std, (val - mean) / std, (test - mean) / std


def make_windows(data, pred_len: int):
    T, C = data.shape
    n = T - SEQ_LEN - pred_len + 1
    if n <= 0:
        raise ValueError(f"Not enough data: T={T}, seq={SEQ_LEN}, pred={pred_len}")
    s0, s1 = data.strides
    view = np.lib.stride_tricks.as_strided(
        data, shape=(n, SEQ_LEN + pred_len, C), strides=(s0, s0, s1))
    xs = np.ascontiguousarray(view[:, :SEQ_LEN])
    ys = np.ascontiguousarray(view[:, SEQ_LEN:])
    return torch.tensor(xs, dtype=torch.float32), torch.tensor(ys, dtype=torch.float32)


RAW_DATA = load_etth1()
N_VARIATES = RAW_DATA.shape[1]
assert RAW_DATA.shape[0] == EXPECTED_T_TOTAL, (
    f"ETTh1 length {RAW_DATA.shape[0]} != committed {EXPECTED_T_TOTAL}; wrong file attached?")
assert N_VARIATES == 7, f"Expected 7 ETTh1 variates, got {N_VARIATES}"

# ── Protocol tripwire: fail in seconds, never partway into the wrong split ──
_tr, _va, _te = split_normalise_etth1(RAW_DATA)
assert len(_tr) == TRAIN_END and len(_va) == VAL_END - TRAIN_END, "Split sizes wrong."
for _H in PRED_LENS:
    _tr_x, _ = make_windows(_tr, _H)
    _te_x, _ = make_windows(_te, _H)
    assert len(_tr_x) == EXPECTED_TRAIN_WINDOWS[_H], (
        f"H={_H}: train windows {len(_tr_x)} != committed {EXPECTED_TRAIN_WINDOWS[_H]}")
    assert len(_te_x) == EXPECTED_TEST_WINDOWS[_H], (
        f"H={_H}: test windows {len(_te_x)} != committed {EXPECTED_TEST_WINDOWS[_H]}")
    _spe = -(-len(_tr_x) // BATCH_SIZE)   # ceil; drop_last=False matches train_etth1.ipynb
    assert _spe == EXPECTED_SPE_B32[_H], (
        f"H={_H}: steps/epoch {_spe} != committed {EXPECTED_SPE_B32[_H]}")
print(f"Protocol tripwires PASS for all horizons {PRED_LENS}: "
      f"train/test windows and steps/epoch match the committed ETTh1 split.")
del _tr, _va, _te, _tr_x, _te_x


In [ ]:
# ── Cell 5: Model construction from committed modules ────────────────────────
def build_arm(mode: str, pred_len: int):
    return M.build_model(
        mode, seq_len=SEQ_LEN, pred_len=pred_len, num_variates=N_VARIATES,
        patch_size=PATCH_SIZE, stride=PATCH_STRIDE, d_model=D_MODEL,
        n_heads=N_HEADS, n_layers=N_LAYERS, dropout=DROPOUT)


def _n_params(m) -> int:
    return sum(p.numel() for p in m.parameters())


N_PATCHES = (SEQ_LEN - PATCH_SIZE) // PATCH_STRIDE + 1
assert N_PATCHES == M.num_patches(SEQ_LEN, PATCH_SIZE, PATCH_STRIDE)

# Architecture asserts at every horizon: head Linear(N_PATCHES*D_MODEL, H); CI/CD param equality.
_x = torch.zeros(2, SEQ_LEN, N_VARIATES)
for _H in PRED_LENS:
    _models = {mode: build_arm(mode, _H) for mode in ("CI", "CD")}
    for _mode, _m in _models.items():
        _y = _m(_x)
        assert _y.shape == (2, _H, N_VARIATES), f"{_mode} H={_H} wrong output shape: {_y.shape}"
        assert _m.head.in_features == N_PATCHES * D_MODEL, f"{_mode} head wrong: {_m.head}"
        assert _m.head.out_features == _H, f"{_mode} head wrong: {_m.head}"
    assert _n_params(_models["CI"]) == _n_params(_models["CD"]), f"CI/CD param mismatch at H={_H}"
    del _models
print(f"Arms OK for all horizons {PRED_LENS}: N_PATCHES={N_PATCHES}, "
      f"head Linear({N_PATCHES * D_MODEL}, pred_len)")
del _x, _y
free_cuda()


In [ ]:
# ── Cell 6: Training engine with per-epoch atomic checkpoint+resume ─────────
WORK         = Path("/kaggle/working")
OUT_PATH     = WORK / "results_etth1_b4.csv"
WINDOWS_PATH = WORK / "results_etth1_b4_windows.csv"

# ── Cross-session bootstrap (three artefact classes now) ────────────────────
_prior_main = [p for p in INPUT_ROOT.rglob("results_etth1_b4.csv")]
if not OUT_PATH.exists() and _prior_main:
    assert len(_prior_main) == 1, f"Multiple prior main registries attached: {_prior_main}."
    shutil.copy(_prior_main[0], OUT_PATH)
    print(f"[bootstrap] main registry seeded from {_prior_main[0]}")

print(f"[convention] window_start = VAL_END + i for the i-th 0-indexed test window "
      f"(VAL_END={VAL_END}) -- confirm this matches partition_etth1_coupling.py's own "
      f"convention before running the subgroup join; logged here so that check does "
      f"not depend on re-reading this notebook's source.")

_prior_windows = [p for p in INPUT_ROOT.rglob("results_etth1_b4_windows.csv")]
if not WINDOWS_PATH.exists() and _prior_windows:
    assert len(_prior_windows) == 1, f"Multiple prior window registries attached: {_prior_windows}."
    shutil.copy(_prior_windows[0], WINDOWS_PATH)
    print(f"[bootstrap] window registry seeded from {_prior_windows[0]}")

_prior_ckpts = {}
for _p in INPUT_ROOT.rglob("ckpt_ETTh1B4_*.pt"):
    _prior_ckpts.setdefault(_p.name, []).append(_p)
for _name, _srcs in sorted(_prior_ckpts.items()):
    assert len(_srcs) == 1, f"Checkpoint {_name} found in multiple attached inputs: {_srcs}."
    if not (WORK / _name).exists():
        shutil.copy(_srcs[0], WORK / _name)
        print(f"[bootstrap] {_name} seeded from {_srcs[0]}")

# ── Session budget (mechanism identical; estimates unmeasured) ──────────────
SESSION_T0 = time.time()
EPOCH_EST_S = {"CD": 300.0, "CI": 150.0}   # placeholder; self-corrects from measured epochs
_BUDGET_MARGIN_S = 900.0


def _out_of_budget(mode: str) -> bool:
    elapsed = time.time() - SESSION_T0
    return elapsed + EPOCH_EST_S[mode] + _BUDGET_MARGIN_S > SESSION_BUDGET_S


def _atomic_torch_save(obj, path):
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)


def _atomic_csv_save(rows, path, schema):
    tmp = path.with_suffix(".csv.tmp")
    pd.DataFrame(rows, columns=schema).to_csv(tmp, index=False)
    os.replace(tmp, path)


def _ckpt_path(pred_len: int, mode: str, seed: int):
    return WORK / f"ckpt_ETTh1B4_h{pred_len}_{mode}_s{seed}.pt"


def _rng_capture():
    return {
        "py": random.getstate(), "np": np.random.get_state(),
        "torch": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }


def _rng_restore(st):
    random.setstate(st["py"])
    np.random.set_state(st["np"])
    torch.set_rng_state(st["torch"].cpu() if torch.is_tensor(st["torch"]) else st["torch"])
    if st["cuda"] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([t.cpu() if torch.is_tensor(t) else t for t in st["cuda"]])


@torch.no_grad()
def _evaluate(model, loader):
    # Committed ETTh1 convention: NO autocast at eval (differs from the
    # boundary protocol's eval-under-autocast; each family keeps its own
    # committed eval mode).
    model.eval()
    mse = mae = n = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        pred = model(xb)
        mse += nn.functional.mse_loss(pred, yb, reduction="sum").item()
        mae += nn.functional.l1_loss(pred, yb, reduction="sum").item()
        n += yb.numel()
    return mse / n, mae / n


@torch.no_grad()
def _evaluate_per_window(model, loader, start_offset: int):
    # Per-window mean squared/absolute error, in loader order (shuffle=False),
    # keyed by window_start = start_offset + window_index. start_offset is the
    # ABSOLUTE ETTh1 timestep at which the test slice begins (VAL_END) — see
    # the window_start convention note in the header markdown (UNVERIFIED).
    model.eval()
    rows, idx = [], 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        pred = model(xb)
        se = ((pred - yb) ** 2).mean(dim=(1, 2))
        ae = (pred - yb).abs().mean(dim=(1, 2))
        for j in range(se.shape[0]):
            rows.append({"window_start": start_offset + idx,
                         "sq_err": se[j].item(), "abs_err": ae[j].item()})
            idx += 1
    return rows


def _cosine_warmup(optimizer, epoch: int) -> None:
    # Committed ETTh1 schedule: linear warmup, cosine decay to 0 (NO floor —
    # differs from the boundary protocol's 1e-6 floor; kept verbatim per family).
    if epoch < WARMUP_EPOCHS:
        lr = LR * (epoch + 1) / WARMUP_EPOCHS
    else:
        p = (epoch - WARMUP_EPOCHS) / max(1, MAX_EPOCHS - WARMUP_EPOCHS)
        lr = LR * 0.5 * (1.0 + math.cos(math.pi * p))
    for pg in optimizer.param_groups:
        pg["lr"] = lr


In [ ]:
# ── Cell 6b: train_one — resumable at epoch granularity ─────────────────────
def train_one(pred_len: int, mode: str, seed: int):
    # One full run at the committed ETTh1 protocol, resumable at epoch granularity.
    # Returns (main_row, window_rows), or None if the session budget was reached
    # before the run could finish (checkpoint stays on disk for the next session).
    ckpt_file = _ckpt_path(pred_len, mode, seed)
    # Fingerprint includes the schedule hyperparameters, not just the run identity:
    # if WARMUP_EPOCHS/MAX_EPOCHS/PATIENCE/MIN_EPOCHS/LR/WEIGHT_DECAY/GRAD_CLIP are
    # edited between sessions of the same resumed run, this fails loud on resume
    # instead of silently splicing two different protocols into one trajectory.
    fingerprint = {
        "pred_len": pred_len, "mode": mode, "seed": seed, "batch": BATCH_SIZE,
        "warmup_epochs": WARMUP_EPOCHS, "max_epochs": MAX_EPOCHS,
        "patience": PATIENCE, "min_epochs": MIN_EPOCHS,
        "lr": LR, "weight_decay": WEIGHT_DECAY, "grad_clip": GRAD_CLIP,
    }

    set_seed(seed)
    tr, va, te = split_normalise_etth1(RAW_DATA)
    x_tr, y_tr = make_windows(tr, pred_len)
    x_va, y_va = make_windows(va, pred_len)
    x_te, y_te = make_windows(te, pred_len)
    train_ds = TensorDataset(x_tr, y_tr)
    val_dl   = DataLoader(TensorDataset(x_va, y_va), batch_size=BATCH_SIZE, shuffle=False)
    test_dl  = DataLoader(TensorDataset(x_te, y_te), batch_size=BATCH_SIZE, shuffle=False)
    set_seed(seed)   # re-seed so model init is identical to any retry (train_etth1.ipynb convention)

    model  = build_arm(mode, pred_len).to(DEVICE)
    opt    = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = GradScaler("cuda", enabled=(DEVICE.type == "cuda"))
    criterion = nn.MSELoss()
    spe = None

    start_epoch, best_val, best_epoch = 0, float("inf"), 0
    best_total_steps, patience_ctr, steps_so_far = 0, 0, 0
    best_sd = None

    if ckpt_file.exists():
        ck = torch.load(ckpt_file, map_location="cpu", weights_only=False)
        assert ck["fingerprint"] == fingerprint, (
            f"Checkpoint {ckpt_file.name} fingerprint {ck['fingerprint']} does not match "
            f"this run {fingerprint}; refusing to load.")
        model.load_state_dict(ck["model_sd"])
        opt.load_state_dict(ck["opt_sd"])
        scaler.load_state_dict(ck["scaler_sd"])
        _rng_restore(ck["rng"])
        start_epoch      = ck["epoch_done"]
        best_val         = ck["best_val"]
        best_epoch       = ck["best_epoch"]
        best_total_steps = ck["best_total_steps"]
        patience_ctr     = ck["patience_ctr"]
        steps_so_far     = ck["steps_so_far"]
        best_sd          = ck["best_sd"]
        if ck.get("stopped", False):
            start_epoch = MAX_EPOCHS
        print(f"    [resume] {ckpt_file.name}: {min(start_epoch, MAX_EPOCHS)} epochs done, "
              f"best_val={best_val:.6f} @ epoch {best_epoch}"
              f"{'  [stopped]' if ck.get('stopped', False) else ''}", flush=True)

    stopped = False
    train_dl = None    # stays None when the loop is skipped (a resumed, already-stopped run)
    for epoch in range(start_epoch, MAX_EPOCHS):
        if _out_of_budget(mode):
            print(f"    [budget] {SESSION_BUDGET_S / 3600:.1f} h session budget cannot fit "
                  f"another {mode} epoch; stopping cleanly (checkpoint on disk).", flush=True)
            return None
        t0 = time.time()
        _cosine_warmup(opt, epoch)
        shuffle_gen = torch.Generator()
        shuffle_gen.manual_seed(1_000_003 * seed + epoch)
        train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              drop_last=False, generator=shuffle_gen)   # committed ETTh1 protocol
        if spe is None:
            spe = len(train_dl)

        model.train()
        for xb, yb in train_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with autocast("cuda", enabled=(DEVICE.type == "cuda")):
                loss = criterion(model(xb), yb)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(opt)
            scaler.update()
            steps_so_far += 1

        val_mse, _ = _evaluate(model, val_dl)
        if not math.isfinite(val_mse):
            # Declared deviation from train_etth1.ipynb (safety-only; never
            # fires on a finite trajectory): stop cleanly instead of letting
            # NaNs propagate into a saved checkpoint.
            print(f"    [warn] non-finite val at epoch {epoch + 1}; stopping run.", flush=True)
            stopped = True
        elif val_mse < best_val:
            best_val, best_epoch = val_mse, epoch + 1
            best_total_steps = steps_so_far
            patience_ctr = 0
            best_sd = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            # MIN_EPOCHS gate: committed ETTh1 protocol, prevents a degenerate
            # epoch-1 stop before the warmup schedule has peaked.
            if epoch + 1 >= MIN_EPOCHS:
                patience_ctr += 1
                if patience_ctr >= PATIENCE:
                    stopped = True

        _atomic_torch_save({
            "fingerprint": fingerprint, "epoch_done": epoch + 1,
            "model_sd": {k: v.detach().cpu() for k, v in model.state_dict().items()},
            "opt_sd": opt.state_dict(), "scaler_sd": scaler.state_dict(),
            "rng": _rng_capture(), "best_val": best_val, "best_epoch": best_epoch,
            "best_total_steps": best_total_steps, "patience_ctr": patience_ctr,
            "steps_so_far": steps_so_far, "best_sd": best_sd, "stopped": stopped,
        }, ckpt_file)
        epoch_s = time.time() - t0
        EPOCH_EST_S[mode] = max(EPOCH_EST_S[mode], epoch_s)
        print(f"    epoch {epoch + 1:3d}/{MAX_EPOCHS}  val={val_mse:.6f}  "
              f"best={best_val:.6f}@{best_epoch}  ({epoch_s:.0f}s)", flush=True)
        if stopped:
            break

    if best_sd is None:
        ckpt_file.unlink(missing_ok=True)
        raise RuntimeError(
            f"No best state for pred_len={pred_len} mode={mode} seed={seed} "
            f"(all epochs non-finite). Run not recorded; checkpoint removed.")
    model.load_state_dict(best_sd)
    test_mse, test_mae = _evaluate(model, test_dl)
    window_rows = _evaluate_per_window(model, test_dl, start_offset=VAL_END)
    for r in window_rows:
        r["mode"], r["pred_len"], r["seed"] = mode, pred_len, seed

    # Oracle self-check: the per-window mean must reconcile with the aggregate
    # test_mse computed independently by _evaluate (same test pass, two separate
    # code paths) -- catches a dim/axis bug in either aggregation without needing
    # an external reference. Relative tolerance: two different float32 summation
    # orders over ~thousands of windows can disagree at the 1e-5-1e-4 relative
    # level even when both are correct; a real bug (wrong axis, wrong offset) is
    # typically off by a large factor, not a rounding-sized amount.
    _window_mean_mse = sum(r["sq_err"] for r in window_rows) / len(window_rows)
    _rel_gap = abs(_window_mean_mse - test_mse) / max(abs(test_mse), 1e-8)
    assert _rel_gap < 1e-3, (
        f"Per-window mean sq_err ({_window_mean_mse:.8f}) does not reconcile with "
        f"aggregate test_mse ({test_mse:.8f}), rel_gap={_rel_gap:.2e}, for "
        f"pred_len={pred_len} mode={mode} seed={seed}.")

    row = {"mode": mode, "pred_len": pred_len, "seed": seed,
           "test_mse": round(test_mse, 8), "test_mae": round(test_mae, 8),
           "best_epoch": best_epoch, "batch_size": BATCH_SIZE,
           "steps_per_epoch": spe if spe is not None else len(train_ds) // BATCH_SIZE,
           "total_steps_to_best": best_total_steps,
           "warmup_epochs": WARMUP_EPOCHS, "min_epochs": MIN_EPOCHS, "max_epochs": MAX_EPOCHS}

    del model, opt, scaler, train_dl, train_ds, val_dl, test_dl, best_sd
    free_cuda()
    ckpt_file.unlink(missing_ok=True)
    return row, window_rows


In [ ]:
# ── Cell 7: Main sweep — idempotent registry, config-ordered ────────────────
def _key(pred_len: int, mode: str, seed: int) -> tuple:
    return (int(pred_len), str(mode), int(seed))


if OUT_PATH.exists() and OUT_PATH.stat().st_size > 100:
    _existing = pd.read_csv(OUT_PATH)
    assert list(_existing.columns) == SCHEMA_MAIN, (
        f"{OUT_PATH.name} columns {list(_existing.columns)} != expected {SCHEMA_MAIN}; "
        f"refusing to resume (a rewrite would silently drop or reorder columns).")
    results = _existing.to_dict("records")
    done = {_key(r["pred_len"], r["mode"], r["seed"]) for r in results}
    print(f"Registry: {len(done)}/{len(RUN_LIST)} assigned runs already complete.")
else:
    results, done = [], set()

if WINDOWS_PATH.exists() and WINDOWS_PATH.stat().st_size > 100:
    window_results = pd.read_csv(WINDOWS_PATH).to_dict("records")
else:
    window_results = []

fails = []
for idx, (pred_len, mode, seed) in enumerate(RUN_LIST, 1):
    key = _key(pred_len, mode, seed)
    if key in done:
        print(f"[{idx}/{len(RUN_LIST)}] SKIP pred_len={pred_len} mode={mode} seed={seed}")
        continue
    print(f"[{idx}/{len(RUN_LIST)}] pred_len={pred_len} mode={mode} seed={seed}", flush=True)
    t0 = time.time()
    try:
        outcome = train_one(pred_len, mode, seed)
    except Exception as exc:
        free_cuda()
        print(f"  FAILED: {type(exc).__name__}: {exc}", flush=True)
        fails.append((pred_len, mode, seed, repr(exc)))
        continue
    if outcome is None:
        print("  Session budget reached. Next session: new version, attach THIS "
              "version's output as an input, Run All to resume.", flush=True)
        break
    row, w_rows = outcome
    results.append(row)
    window_results.extend(w_rows)
    done.add(key)
    _atomic_csv_save(results, OUT_PATH, SCHEMA_MAIN)
    _atomic_csv_save(window_results, WINDOWS_PATH, SCHEMA_WINDOWS)
    print(f"  DONE mse={row['test_mse']:.4f} epoch={row['best_epoch']} "
          f"windows={len(w_rows)} ({(time.time() - t0) / 3600:.2f} h)", flush=True)

print(f"\nSession end: {len(done)}/{len(RUN_LIST)} runs complete -> {OUT_PATH}")
print(f"Window rows: {len(window_results)} -> {WINDOWS_PATH}")
if fails:
    print(f"{len(fails)} failures (rerun Run All to retry):")
    for f in fails:
        print(" ", f)


In [ ]:
# ── Cell 8: Sanity summary ───────────────────────────────────────────────────
if OUT_PATH.exists():
    df = pd.read_csv(OUT_PATH)
    print(df.to_string(index=False))
    if {"CI", "CD"} <= set(df["mode"]):
        piv = df.groupby(["pred_len", "mode"])["test_mse"].mean().unstack()
        if {"CI", "CD"} <= set(piv.columns):
            piv["CD/CI"] = piv["CD"] / piv["CI"]
        print()
        print(piv.to_string())
else:
    print("No results yet.")

if WINDOWS_PATH.exists():
    wdf = pd.read_csv(WINDOWS_PATH)
    print(f"\nWindow rows: {len(wdf)}  by (mode, pred_len):")
    print(wdf.groupby(["mode", "pred_len"]).size().unstack(fill_value=0).to_string())


In [ ]:
from IPython.display import FileLink, display
display(FileLink("results_etth1_b4.csv"))
display(FileLink("results_etth1_b4_windows.csv"))
